In [14]:
import optuna
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier

c:\Users\Vik\Documents\UGent\Masterproef\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
train_df = pd.read_parquet("data/df_train_preprocessed.parquet")
test_df = pd.read_parquet("data/df_test_preprocessed.parquet")

id_cols = ["month_decision", "weekday_decision", "WEEK_NUM", "case_id"]

train_df.drop(columns=id_cols, inplace=True)
test_df.drop(columns=id_cols, inplace=True)

print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

Train shape: (1088836, 202), Test shape: (437823, 202)


In [16]:
train_df

,target,assignmentdate_238D,birthdate_574D,dateofbirth_337D,days120_123L,days30_165L,description_5085714M,education_1103M,maritalst_385M,pmtaverage_3A,...,max_credlmt_935A_missing,max_dpdmax_757P_missing,max_outstandingamount_362A_missing,max_totaldebtoverduevalue_718A_missing,max_numberofoverdueinstlmaxdat_641D_missing,max_annualeffectiverate_199L_missing,max_annualeffectiverate_63L_missing,mode_role_1084L_missing,mode_type_25L_missing,max_amount_416A_missing
0,0,0.077125,0.091781,0.125197,-0.310926,-0.565845,-6.409159,-2.173400,-2.411442,-0.129158,...,1,1,1,1,1,1,1,0,0,1
1,0,0.077125,0.091781,0.125197,-0.310926,-0.565845,-6.409159,-2.173400,-2.411442,-0.129158,...,1,1,1,1,1,1,1,0,0,1
2,0,0.077125,0.091781,0.125197,-0.310926,-0.565845,-6.409159,-2.173400,-2.411442,-0.129158,...,1,1,1,1,1,1,1,0,0,1
3,0,0.077125,0.091781,0.125197,-0.310926,-0.565845,-6.409159,-2.173400,-2.411442,-0.129158,...,1,1,1,1,1,1,1,0,0,1
4,1,0.077125,0.091781,0.125197,-0.310926,-0.565845,-6.409159,-2.173400,-2.411442,-0.129158,...,1,1,1,1,1,1,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1479675,0,0.077125,0.091781,-0.147920,0.230528,-0.565845,0.156027,-0.702132,-0.029897,-0.129158,...,1,0,0,0,1,1,1,0,0,1
1479676,0,0.077125,0.091781,0.656487,3.479255,2.010193,0.156027,0.797013,0.821998,-0.129158,...,0,0,0,0,1,1,1,0,0,1
1479677,0,0.077125,0.091781,-1.736807,1.313437,-0.565845,0.156027,-0.702132,-0.029897,-0.129158,...,1,0,0,0,1,1,0,0,0,1
1479678,0,0.077125,0.091781,1.051976,-0.852381,-0.565845,0.156027,0.797013,0.821998,-0.129158,...,1,0,1,0,1,1,1,0,0,1


In [17]:
test_df

,target,assignmentdate_238D,birthdate_574D,dateofbirth_337D,days120_123L,days30_165L,description_5085714M,education_1103M,maritalst_385M,pmtaverage_3A,...,max_credlmt_935A_missing,max_dpdmax_757P_missing,max_outstandingamount_362A_missing,max_totaldebtoverduevalue_718A_missing,max_numberofoverdueinstlmaxdat_641D_missing,max_annualeffectiverate_199L_missing,max_annualeffectiverate_63L_missing,mode_role_1084L_missing,mode_type_25L_missing,max_amount_416A_missing
42469,0,0.077125,0.091781,1.335056,2.396346,2.010193,0.156027,-0.702132,-0.029897,-0.129158,...,0,0,0,0,1,0,1,0,0,1
42496,0,0.077125,0.091781,0.906429,0.230528,-0.565845,0.156027,0.797013,0.821998,-0.129158,...,0,1,1,1,0,1,1,0,0,1
42514,0,0.077125,0.091781,1.004976,0.230528,-0.565845,0.156027,0.797013,0.821998,-0.129158,...,0,1,1,1,1,1,1,0,0,1
42521,0,0.077125,0.091781,0.125197,-0.310926,-0.565845,0.156027,0.797013,0.821998,-0.129158,...,1,1,1,1,1,1,1,0,0,1
42533,0,0.077125,0.091781,1.545579,2.396346,3.298212,0.156027,0.797013,-1.510961,-0.129158,...,0,0,0,0,1,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1526654,0,0.077125,0.091781,-1.367308,-0.852381,-0.565845,-6.409159,0.797013,0.821998,-0.129158,...,1,0,0,0,1,1,1,0,0,1
1526655,0,0.077125,0.091781,-2.092444,-0.852381,-0.565845,-6.409159,0.797013,0.821998,-0.129158,...,1,0,0,0,1,1,1,0,0,1
1526656,0,0.077125,0.091781,0.023618,0.230528,-0.565845,-6.409159,0.797013,0.821998,-0.129158,...,0,0,1,0,0,1,1,0,0,1
1526657,0,0.077125,0.091781,-2.101540,0.230528,0.722174,-6.409159,-0.702132,-0.029897,-0.129158,...,1,0,0,0,1,1,1,0,0,0


In [18]:
# List of cols we used
cols = [col for col in train_df.columns if col != 'target']

cols_df = pd.DataFrame({'column_name': cols})

display(cols_df)

,column_name
0,assignmentdate_238D
1,birthdate_574D
2,dateofbirth_337D
3,days120_123L
4,days30_165L
...,...
196,max_annualeffectiverate_199L_missing
197,max_annualeffectiverate_63L_missing
198,mode_role_1084L_missing
199,mode_type_25L_missing


In [19]:
# Define features and target variable
X_train = train_df.drop(columns=["target"])
y_train = train_df["target"]

X_test = test_df.drop(columns=["target"])
y_test = test_df["target"]

## 3.1 Logistische regressie

In [20]:
# Kostenverhouding: een gemiste defaulter (lening gegeven, niet terugbetaald) kost
# COST_FN keer zoveel als een onterecht geweigerde goede klant (COST_FP)
COST_FN = 5  # missen van een defaulter (target=1 fout voorspeld als 0)
# onterecht weigeren van een goede klant (target=0 fout voorspeld als 1)
COST_FP = 1

class_weight = {0: COST_FP, 1: COST_FN}

model_cs = LogisticRegression(max_iter=1000, class_weight=class_weight)
model_cs.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*","{0: 1, 1: 5}"
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For 

In [21]:
y_pred_cs = model_cs.predict(X_test)
y_pred_proba_cs = model_cs.predict_proba(X_test)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred_cs):.4f}")
print(f"AUC:       {roc_auc_score(y_test, y_pred_proba_cs):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_cs):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_cs):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred_cs):.4f}")

tn, fp, fn, tp = confusion_matrix(y_test, y_pred_cs).ravel()
print(f"\nConfusion matrix (cost-sensitive):")
print(f"TN: {tn}\tFP: {fp}")
print(f"FN: {fn}\tTP: {tp}")

Accuracy:  0.9468
AUC:       0.8229
Precision: 0.2082
Recall:    0.2215
F1-score:  0.2147

Confusion matrix (cost-sensitive):
TN: 411341	FP: 12106
FN: 11192	TP: 3184


### 3.1.1 Hyperparameter tuning met Optuna (temporele CV)

Onze data heeft een temporele dimensie (`WEEK_NUM`, gebruikt voor de train/test-split in de data preparation stap). Een gewone `KFold`/`StratifiedKFold` schudt de rijen willekeurig door elkaar, waardoor een validatiefold observaties uit *vroegere* weken kan bevatten dan rijen waarop het model in diezelfde fold getraind is. Dat is train/test-lekkage in de tijd: het model "ziet" toekomstige informatie (bv. verschuivingen in klantgedrag, macro-economische trends) die het in productie nooit vooraf zou hebben, en de CV-score is te optimistisch.

**Alternatief: `TimeSeriesSplit`** (walk-forward / expanding window CV). Elke fold traint enkel op observaties die chronologisch vóór de validatiefold liggen, net zoals bij de echte train/test-split. Zo blijft de evaluatie tijdens hyperparameter tuning consistent met hoe het model in productie zou werken.

In [22]:
# WEEK_NUM werd bij het inladen als id-kolom verwijderd (zie hierboven), maar
# hebben we hier opnieuw nodig om de trainrijen chronologisch te ordenen zodat
# TimeSeriesSplit echte "train op verleden, valideer op toekomst"-folds maakt.
week_num_train = pd.read_parquet(
    "data/df_train_preprocessed.parquet", columns=["WEEK_NUM"]
)["WEEK_NUM"]

chrono_order = week_num_train.sort_values(kind="stable").index
X_train_sorted = X_train.loc[chrono_order].reset_index(drop=True)
y_train_sorted = y_train.loc[chrono_order].reset_index(drop=True)

print(f"Train weken: {week_num_train.min()}-{week_num_train.max()}")

Train weken: 0-52


In [ ]:
N_SPLITS = 5  # walk-forward folds
N_TRIALS = 30  # verlaag dit (bv. 10-15) als een run te lang duurt


def objective(trial: optuna.Trial) -> float:
    C = trial.suggest_float("C", 1e-4, 1e2, log=True)
    l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)

    tscv = TimeSeriesSplit(n_splits=N_SPLITS)
    fold_costs = []
    for fold_train_idx, fold_val_idx in tscv.split(X_train_sorted):
        X_tr, X_val = X_train_sorted.iloc[fold_train_idx], X_train_sorted.iloc[fold_val_idx]
        y_tr, y_val = y_train_sorted.iloc[fold_train_idx], y_train_sorted.iloc[fold_val_idx]

        model = LogisticRegression(
            C=C,
            l1_ratio=l1_ratio,
            solver="saga",  # enige solver die l1, l2 en elasticnet alle drie ondersteunt
            class_weight=class_weight,  # zelfde COST_FN/COST_FP-verhouding als hierboven
            max_iter=1000,
        )
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)

        # Optimaliseer voor de echte business-kost i.p.v. AUC/F1, zodat de tuning
        # rechtstreeks aansluit bij de kostenverhouding die we als bedrijf hanteren.
        tn, fp, fn, tp = confusion_matrix(y_val, y_pred, labels=[0, 1]).ravel()
        fold_costs.append((COST_FN * fn + COST_FP * fp) / len(y_val))

    return sum(fold_costs) / len(fold_costs)


study = optuna.create_study(
    direction="minimize", sampler=optuna.samplers.TPESampler(seed=42)
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("Beste parameters:", study.best_params)
print(f"Beste gemiddelde kost per observatie (CV): {study.best_value:.4f}")

[I 2026-07-19 22:25:15,343] A new study created in memory with name: no-name-2e9fc0a2-d4c4-4abb-868b-13665b08142a
  0%|          | 0/30 [00:00<?, ?it/s]c:\Users\Vik\Documents\UGent\Masterproef\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Vik\Documents\UGent\Masterproef\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=None. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\Vik\Documents\UGent\Masterproef\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoi

[I 2026-07-19 22:27:35,941] Trial 0 finished with value: 0.15404690530770587 and parameters: {'C': 0.017670169402947963, 'penalty': 'l1'}. Best is trial 0 with value: 0.15404690530770587.


c:\Users\Vik\Documents\UGent\Masterproef\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Vik\Documents\UGent\Masterproef\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Vik\Documents\UGent\Masterproef\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and w

[I 2026-07-19 22:28:41,315] Trial 1 finished with value: 0.15388379474519484 and parameters: {'C': 0.0008632008168602544, 'penalty': 'elasticnet', 'l1_ratio': 0.6011150117432088}. Best is trial 1 with value: 0.15388379474519484.


: 

: 

In [ ]:
best_params = study.best_params
model_cs_tuned = LogisticRegression(
    C=best_params["C"],
    penalty=best_params["penalty"],
    l1_ratio=best_params.get("l1_ratio"),
    solver="saga",
    class_weight=class_weight,
    max_iter=1000,
)
model_cs_tuned.fit(X_train, y_train)

y_pred_cs_tuned = model_cs_tuned.predict(X_test)
y_pred_proba_cs_tuned = model_cs_tuned.predict_proba(X_test)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred_cs_tuned):.4f}")
print(f"AUC:       {roc_auc_score(y_test, y_pred_proba_cs_tuned):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_cs_tuned):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_cs_tuned):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred_cs_tuned):.4f}")

tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, y_pred_cs_tuned).ravel()
print(f"\nConfusion matrix (LR getuned, cost-sensitive):")
print(f"TN: {tn_t}\tFP: {fp_t}")
print(f"FN: {fn_t}\tTP: {tp_t}")

## 3.2 XGBoost

In [ ]:
# # Zelfde kostenverhouding als bij de logistische regressie (COST_FN / COST_FP),
# # via scale_pos_weight: het gewicht van de positieve klasse (target=1) t.o.v. de negatieve.
# model_xgb_cs = XGBClassifier(
#     n_estimators=300,
#     max_depth=4,
#     learning_rate=0.1,
#     scale_pos_weight=COST_FN / COST_FP,
#     eval_metric="logloss",
# )
# model_xgb_cs.fit(X_train, y_train)

In [ ]:
# y_pred_xgb_cs = model_xgb_cs.predict(X_test)
# y_pred_proba_xgb_cs = model_xgb_cs.predict_proba(X_test)[:, 1]

# print(f"Accuracy:  {accuracy_score(y_test, y_pred_xgb_cs):.4f}")
# print(f"AUC:       {roc_auc_score(y_test, y_pred_proba_xgb_cs):.4f}")
# print(f"Precision: {precision_score(y_test, y_pred_xgb_cs):.4f}")
# print(f"Recall:    {recall_score(y_test, y_pred_xgb_cs):.4f}")
# print(f"F1-score:  {f1_score(y_test, y_pred_xgb_cs):.4f}")

# tn_xgb, fp_xgb, fn_xgb, tp_xgb = confusion_matrix(
#     y_test, y_pred_xgb_cs).ravel()
# print(f"\nConfusion matrix (XGBoost, cost-sensitive):")
# print(f"TN: {tn_xgb}\tFP: {fp_xgb}")
# print(f"FN: {fn_xgb}\tTP: {tp_xgb}")

## 3.3 MLP

In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# X_train_t = torch.tensor(X_train.values, dtype=torch.float32).to(device)
# y_train_t = torch.tensor(
#     y_train.values, dtype=torch.float32).unsqueeze(1).to(device)
# X_test_t = torch.tensor(X_test.values, dtype=torch.float32).to(device)

# train_dataset = torch.utils.data.TensorDataset(X_train_t, y_train_t)
# train_loader = torch.utils.data.DataLoader(
#     train_dataset, batch_size=1024, shuffle=True)

In [ ]:
# # Zeer simpel MLP: 1 hidden layer
# class SimpleMLP(nn.Module):
#     def __init__(self, n_features, n_hidden=64):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(n_features, n_hidden),
#             nn.ReLU(),
#             nn.Linear(n_hidden, 1),
#         )

#     def forward(self, x):
#         return self.net(x)


# model_mlp_cs = SimpleMLP(n_features=X_train_t.shape[1]).to(device)

# # Zelfde kostenverhouding als bij logistic regression / XGBoost, via pos_weight op de BCE-loss
# pos_weight = torch.tensor([COST_FN / COST_FP], device=device)
# criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
# optimizer = torch.optim.Adam(model_mlp_cs.parameters(), lr=1e-3)

# n_epochs = 10
# for epoch in range(n_epochs):
#     model_mlp_cs.train()
#     epoch_loss = 0.0
#     for X_batch, y_batch in train_loader:
#         optimizer.zero_grad()
#         logits = model_mlp_cs(X_batch)
#         loss = criterion(logits, y_batch)
#         loss.backward()
#         optimizer.step()
#         epoch_loss += loss.item() * X_batch.size(0)

#     epoch_loss /= len(train_dataset)
#     print(f"Epoch {epoch + 1}/{n_epochs} - loss: {epoch_loss:.4f}")

In [ ]:
# model_mlp_cs.eval()
# with torch.no_grad():
#     y_pred_proba_mlp_cs = torch.sigmoid(
#         model_mlp_cs(X_test_t)).cpu().numpy().ravel()
# y_pred_mlp_cs = (y_pred_proba_mlp_cs >= 0.5).astype(int)

# print(f"Accuracy:  {accuracy_score(y_test, y_pred_mlp_cs):.4f}")
# print(f"AUC:       {roc_auc_score(y_test, y_pred_proba_mlp_cs):.4f}")
# print(f"Precision: {precision_score(y_test, y_pred_mlp_cs):.4f}")
# print(f"Recall:    {recall_score(y_test, y_pred_mlp_cs):.4f}")
# print(f"F1-score:  {f1_score(y_test, y_pred_mlp_cs):.4f}")

# tn_mlp, fp_mlp, fn_mlp, tp_mlp = confusion_matrix(
#     y_test, y_pred_mlp_cs).ravel()
# print(f"\nConfusion matrix (MLP, cost-sensitive):")
# print(f"TN: {tn_mlp}\tFP: {fp_mlp}")
# print(f"FN: {fn_mlp}\tTP: {tp_mlp}")